In [ ]:
!pip install --upgrade "mediapipe==0.10.35" "protobuf>=5.29.5,<6" -q

import re
import sys
from pathlib import Path

# Python 3.13 on Kaggle does not bind the trailing name of
# `import mediapipe...core`, so MediaPipe's audio package raises
# NameError: name 'core' is not defined.
_IMPORT_LINE = re.compile(r"^import (mediapipe(?:\.\w+)+)$", re.M)
for _init_path in Path("/usr/local/lib").glob("python*/dist-packages/mediapipe/tasks/python/**/__init__.py"):
    _text = _init_path.read_text()
    _patched = _IMPORT_LINE.sub(
        lambda match: f"import {match.group(1)} as {match.group(1).rsplit('.', 1)[-1]}",
        _text,
    )
    if _patched != _text:
        _init_path.write_text(_patched)

for _name in list(sys.modules):
    if _name == "mediapipe" or _name.startswith("mediapipe."):
        del sys.modules[_name]

import os
import cv2
import numpy as np
import mediapipe as mp
import shutil
from glob import glob
from tqdm import tqdm
from pathlib import Path

BODY_LANDMARKS = [
    "nose", "leftEyeInner", "leftEye", "leftEyeOuter", "rightEyeInner", "rightEye", "rightEyeOuter",
    "leftEar", "rightEar", "mouthLeft", "mouthRight", "leftShoulder", "rightShoulder",
    "leftElbow", "rightElbow", "leftWrist", "rightWrist", "leftPinky", "rightPinky",
    "leftIndex", "rightIndex", "leftThumb", "rightThumb",
    "leftHip", "rightHip", "leftKnee", "rightKnee", "leftAnkle", "rightAnkle",
    "leftHeel", "rightHeel", "leftFootIndex", "rightFootIndex", "neck"
]

HAND_LANDMARKS = [
    "wrist", "indexTip", "indexDIP", "indexPIP", "indexMCP",
    "middleTip", "middleDIP", "middlePIP", "middleMCP",
    "ringTip", "ringDIP", "ringPIP", "ringMCP",
    "littleTip", "littleDIP", "littlePIP", "littleMCP",
    "thumbTip", "thumbIP", "thumbMP", "thumbCMC",
]

HANDS_LANDMARKS = [name + suffix for name in HAND_LANDMARKS for suffix in ["_0", "_1"]]
LANDMARKS = BODY_LANDMARKS + HANDS_LANDMARKS

print(f"Da cau hinh danh muc. Tong: {len(LANDMARKS)} diem khop (34 Body + 42 Hands).")


In [ ]:
import logging
import os
import urllib.request
import cv2
import numpy as np
import mediapipe as mp

logger = logging.getLogger("keypoints")
if not logger.handlers:
    logging.basicConfig(level=logging.INFO, format="%(asctime)s %(levelname)s %(message)s")

# Keep OpenCV from starting a thread pool before the process pool forks.
cv2.setNumThreads(1)

BaseOptions = mp.tasks.BaseOptions
PoseLandmarker = mp.tasks.vision.PoseLandmarker
PoseLandmarkerOptions = mp.tasks.vision.PoseLandmarkerOptions
HandLandmarker = mp.tasks.vision.HandLandmarker
HandLandmarkerOptions = mp.tasks.vision.HandLandmarkerOptions
VisionRunningMode = mp.tasks.vision.RunningMode

# Full pose model matches the old Holistic model_complexity=1.
# IMAGE mode has no tracker state, so one landmarker can serve every clip in a worker.
MODEL_DIR = "/kaggle/working/mediapipe_models" if os.path.isdir("/kaggle") else os.path.join(os.getcwd(), "mediapipe_models")
os.makedirs(MODEL_DIR, exist_ok=True)
POSE_MODEL_PATH = os.path.join(MODEL_DIR, "pose_landmarker_full.task")
HAND_MODEL_PATH = os.path.join(MODEL_DIR, "hand_landmarker.task")
MODEL_URLS = {
    POSE_MODEL_PATH: "https://storage.googleapis.com/mediapipe-models/pose_landmarker/pose_landmarker_full/float16/1/pose_landmarker_full.task",
    HAND_MODEL_PATH: "https://storage.googleapis.com/mediapipe-models/hand_landmarker/hand_landmarker/float16/1/hand_landmarker.task",
}

def _download_models():
    for model_path, url in MODEL_URLS.items():
        if os.path.exists(model_path) and os.path.getsize(model_path) > 0:
            continue
        urllib.request.urlretrieve(url, model_path)

_download_models()

POSE_OPTIONS = PoseLandmarkerOptions(
    base_options=BaseOptions(model_asset_path=POSE_MODEL_PATH),
    running_mode=VisionRunningMode.IMAGE,
    num_poses=1,
    min_pose_detection_confidence=0.5,
    min_pose_presence_confidence=0.5,
    min_tracking_confidence=0.5,
)
HAND_OPTIONS = HandLandmarkerOptions(
    base_options=BaseOptions(model_asset_path=HAND_MODEL_PATH),
    running_mode=VisionRunningMode.IMAGE,
    num_hands=2,
    min_hand_detection_confidence=0.5,
    min_hand_presence_confidence=0.5,
    min_tracking_confidence=0.5,
)

POSE_INDEX = {
    "nose": 0, "leftEyeInner": 1, "leftEye": 2, "leftEyeOuter": 3, "rightEyeInner": 4, "rightEye": 5, "rightEyeOuter": 6,
    "leftEar": 7, "rightEar": 8, "mouthLeft": 9, "mouthRight": 10, "leftShoulder": 11, "rightShoulder": 12,
    "leftElbow": 13, "rightElbow": 14, "leftWrist": 15, "rightWrist": 16, "leftPinky": 17, "rightPinky": 18,
    "leftIndex": 19, "rightIndex": 20, "leftThumb": 21, "rightThumb": 22, "leftHip": 23, "rightHip": 24,
    "leftKnee": 25, "rightKnee": 26, "leftAnkle": 27, "rightAnkle": 28, "leftHeel": 29, "rightHeel": 30,
    "leftFootIndex": 31, "rightFootIndex": 32,
}
HAND_INDEX = {
    "wrist": 0, "thumbCMC": 1, "thumbMP": 2, "thumbIP": 3, "thumbTip": 4,
    "indexMCP": 5, "indexPIP": 6, "indexDIP": 7, "indexTip": 8,
    "middleMCP": 9, "middlePIP": 10, "middleDIP": 11, "middleTip": 12,
    "ringMCP": 13, "ringPIP": 14, "ringDIP": 15, "ringTip": 16,
    "littleMCP": 17, "littlePIP": 18, "littleDIP": 19, "littleTip": 20,
}
ZERO = (0.0, 0.0, 0.0)
_POSE_LANDMARKER = None
_HAND_LANDMARKER = None

def _select_hands(hand_result):
    chosen = {}
    landmarks_list = hand_result.hand_landmarks or []
    handedness_list = hand_result.handedness or []
    for landmarks, handedness in zip(landmarks_list, handedness_list):
        if not handedness:
            continue
        label = handedness[0].category_name
        score = handedness[0].score
        if label not in ("Left", "Right"):
            continue
        previous = chosen.get(label)
        if previous is None or score > previous[0]:
            chosen[label] = (score, landmarks)
    return chosen

_BODY_ROW = {name: index for index, name in enumerate(BODY_LANDMARKS)}
_NECK_ROW = _BODY_ROW["neck"]
_LEFT_SHOULDER_ROW = _BODY_ROW["leftShoulder"]
_RIGHT_SHOULDER_ROW = _BODY_ROW["rightShoulder"]
# HANDS_LANDMARKS is name-major: wrist_0, wrist_1, indexTip_0, indexTip_1, ...
_HAND_ROW = {
    (name, slot): len(BODY_LANDMARKS) + name_index * 2 + slot
    for name_index, name in enumerate(HAND_LANDMARKS)
    for slot in (0, 1)
}

def _fill_pose(row, pose_result):
    if not pose_result.pose_landmarks:
        return
    landmarks = pose_result.pose_landmarks[0]
    for name, mp_index in POSE_INDEX.items():
        lm = landmarks[mp_index]
        slot = _BODY_ROW[name]
        row[slot, 0] = lm.x
        row[slot, 1] = lm.y
        row[slot, 2] = lm.z
    row[_NECK_ROW] = (row[_LEFT_SHOULDER_ROW] + row[_RIGHT_SHOULDER_ROW]) / 2

def _fill_hand(row, hand_result):
    chosen_hands = _select_hands(hand_result)
    for slot, label in ((0, "Left"), (1, "Right")):
        picked = chosen_hands.get(label)
        if picked is None:
            continue
        landmarks = picked[1]
        for name, mp_index in HAND_INDEX.items():
            lm = landmarks[mp_index]
            joint = _HAND_ROW[(name, slot)]
            row[joint, 0] = lm.x
            row[joint, 1] = lm.y
            row[joint, 2] = lm.z

def extract_sign_language_features(video_path, npy_out_path, pose_landmarker, hand_landmarker):
    cap = cv2.VideoCapture(video_path)
    if not cap.isOpened():
        logger.error("Cannot open video %s", video_path)
        return False

    frames = []
    try:
        while True:
            ret, frame = cap.read()
            if not ret:
                break
            rgb_frame = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
            mp_image = mp.Image(image_format=mp.ImageFormat.SRGB, data=rgb_frame)
            row = np.zeros((len(LANDMARKS), 3), dtype=np.float32)
            _fill_pose(row, pose_landmarker.detect(mp_image))
            _fill_hand(row, hand_landmarker.detect(mp_image))
            frames.append(row)
    finally:
        cap.release()

    if not frames:
        return False

    os.makedirs(os.path.dirname(npy_out_path), exist_ok=True)
    np.save(npy_out_path, np.stack(frames))
    return True

def _init_worker():
    global _POSE_LANDMARKER, _HAND_LANDMARKER
    logging.basicConfig(level=logging.INFO, format="%(asctime)s %(levelname)s %(message)s")
    cv2.setNumThreads(1)
    os.environ["OMP_NUM_THREADS"] = "1"
    _POSE_LANDMARKER = PoseLandmarker.create_from_options(POSE_OPTIONS)
    _HAND_LANDMARKER = HandLandmarker.create_from_options(HAND_OPTIONS)

def _extract_job(job):
    video_path, npy_out_path = job
    try:
        return extract_sign_language_features(video_path, npy_out_path, _POSE_LANDMARKER, _HAND_LANDMARKER)
    except Exception:
        logger.exception("Failed %s", video_path)
        return False

logger.info("Pipeline ready. float32 (T, 76, 3), one model pair per worker.")


In [ ]:
import os
import shutil

FRAME_SPLIT_DIR = "/kaggle/input/datasets/nguyenanfms/vsl-vietnamese-sign-language-v2/processed/processed/frame_splited"
SPLITS = ("train", "test")
OUTPUT_KEYPOINTS_DIR = "/kaggle/working/keypoints_raw"
KEEP_WORKING_ITEMS = {"mediapipe_models"}

if os.path.exists(OUTPUT_KEYPOINTS_DIR):
    shutil.rmtree(OUTPUT_KEYPOINTS_DIR)
    print("Da don sach keypoints_raw cu.")

if os.path.exists("/kaggle/working"):
    for item in os.listdir("/kaggle/working"):
        if item.startswith(".") or item in KEEP_WORKING_ITEMS:
            continue
        item_path = os.path.join("/kaggle/working", item)
        if os.path.isdir(item_path):
            shutil.rmtree(item_path)
        else:
            os.remove(item_path)

split_classes = {}
for split in SPLITS:
    split_dir = os.path.join(FRAME_SPLIT_DIR, split)
    classes = sorted(
        (name for name in os.listdir(split_dir) if os.path.isdir(os.path.join(split_dir, name))),
        reverse=True,
    )
    split_classes[split] = classes
    print(f"{split}: {len(classes)} classes, chay nguoc tu '{classes[0]}' ve '{classes[-1]}'.")


In [ ]:
from glob import glob
from multiprocessing import get_context
from tqdm import tqdm
from pathlib import Path
import os

jobs = []
for split in SPLITS:
    for class_name in split_classes[split]:
        class_input_path = os.path.join(FRAME_SPLIT_DIR, split, class_name)
        for video_path in sorted(glob(os.path.join(class_input_path, "*.mp4")), reverse=True):
            video_id = Path(video_path).stem
            npy_out_path = os.path.join(OUTPUT_KEYPOINTS_DIR, split, class_name, f"{video_id}.npy")
            if os.path.exists(npy_out_path):
                continue
            jobs.append((video_path, npy_out_path))

worker_count = max(1, os.cpu_count() or 1)
logger.info("Extracting %d videos with %d workers", len(jobs), worker_count)

if jobs:
    with get_context("fork").Pool(processes=worker_count, initializer=_init_worker) as pool:
        for _ok in tqdm(pool.imap_unordered(_extract_job, jobs, chunksize=4), total=len(jobs)):
            pass

logger.info("Finished train and test.")


In [ ]:
import os
print("Nen keypoints_raw thanh keypoints_raw.zip...")
!zip -rq /kaggle/working/keypoints_raw.zip /kaggle/working/keypoints_raw
print("Xong. Tai keypoints_raw.zip tu Output cua Kaggle.")
